# Working with a dataset

`get` returns a `Dataset`: a view of one accession and everything seqout knows
about it. This notebook goes through each field, what it holds, and what you
can do with it.

The example is GSE114725, a single-cell study of the breast tumour immune
environment, with 56 samples and 173 sequencing runs.

In [1]:
import pandas as pd

from seqout import connect

sq = connect()
d = sq.get("GSE114725")
d

Dataset('GSE114725', kind='series')

Nothing has been fetched yet. `Dataset` reads a field the first time you ask
for it and keeps the answer, so a field costs one request however often you
use it, and a field you never touch costs nothing.

The fields are:

| field | holds |
|---|---|
| `meta` | the project record: title, summary, design, dates, supplementary files |
| `samples` | one record per sample |
| `experiments` | one record per library preparation |
| `runs` | one record per sequencing run, with its file URLs |
| `pubs` | the publications linked to the dataset |
| `links` | the same data in other archives |
| `enriched` | structured labels for the samples, where seqout has prepared them |
| `detail` | the record for the accession itself, when it names a sample or a run |

and four that say where the data sits: `kind`, `project`, `geo`, `sra`.

## meta — the project record

`meta` is the study description as the archive holds it.

In [2]:
m = d.meta

print(m.title)
print()
print(m.summary[:300], "...")
print()
print("design:    ", (m.overall_design or "")[:120])
print("organisms: ", m.organisms)
print("centre:    ", m.center_name, f"({m.country_code})")
print("published: ", m.published_at, "| updated:", m.updated_at)
print("type:      ", m.series_type)
print("single cell:", m.is_single_cell, "-", m.single_cell_modality)

Single-cell Map of Diverse Immune Phenotypes in the Breast Tumor Microenvironment 3' RNA Sequencing

Knowledge of immune cell phenotypes in the tumor microenvironment is essential for understanding mechanisms of cancer progression and immunotherapy response. We created an immune map of breast cancer using single-cell RNA-seq data from 45,000 immune cells from eight breast carcinomas, as well as mat ...

design:     Single-cell RNA sequencing was performed on eight donors using the InDrop v2 protocol. For each donor populations of CD4
organisms:  ['Homo sapiens']
centre:     Memorial Sloan Kettering Cancer Center (None)
published:  2018-06-27 | updated: 2019-03-26
type:       ['Expression profiling by high throughput sequencing']
single cell: False - None


### Supplementary files

`supplementary_data` lists the processed files the submitter uploaded — count
matrices, annotations, archives — as `(url, type)` pairs. These are usually
what you want when you do not intend to reprocess the raw reads.

In [3]:
for url, kind in m.supplementary_data:
    print(f"{kind:6} {url.rsplit('/', 1)[-1]}")

TAR    GSE114725_RAW.tar
CSV    GSE114725_rna_imputed.csv.gz
CSV    GSE114725_rna_raw.csv.gz


`download_project_supplementary_data` fetches all of them in parallel. It is
not run here because it writes to disk.

```python
from pathlib import Path
sq.download_project_supplementary_data(d.meta, Path("GSE114725"))
```

### Similar datasets

`neighbors` holds the datasets nearest to this one in seqout's embedding of
study text. It is a way to find related work that does not share an author or
a citation.

In [4]:
for n in m.neighbors[:8]:
    print(f"{n.accession:14} {n.source}")

GSE114724      geo
GSE114727      geo
GSE141665      geo
SRP402962      sra
GSE297298      geo
ERP130387      sra
GSE143423      geo
ERP137510      sra


Those accessions go straight back into `get`, or into `summaries` to read all
their titles in one request.

In [5]:
sq.summaries([n.accession for n in m.neighbors[:5]]).to_df()[["accession", "title"]]

,accession,title
0,GSE114724,Single-cell Map of Diverse Immune Phenotypes i...
1,GSE114727,Single-cell Map of Diverse Immune Phenotypes i...
2,GSE141665,Single-cell RNA-sequencing of γδ-T cells from ...
3,SRP402962,Single-cell RNA sequencing of infiltrated immu...
4,GSE297298,Single-cell RNA-sequencing identifies anti-can...


## samples — one record per sample

A GEO or ArrayExpress sample carries channels. Each channel has its organism,
its source material, the protocols used, and a `characteristics` dictionary
whose keys are chosen by the submitter.

In [6]:
s = d.samples[0]

print(s.accession, "-", s.title)
print("type:    ", s.sample_type)
print("platform:", s.platform_ref)

ch = s.channels[0]
print("organism:", [o.text for o in ch.organisms])
print("source:  ", ch.source)
for tag, value in ch.characteristics.items():
    print(f"  {tag}: {value}")

GSM3148585 - BC01_BLOOD1
type:     SRA
platform: GPL16791
organism: ['Homo sapiens']
source:   BC01
  cell type: CD45+ leukocytes
  donor age (years): 38
  resident tissue: blood


Because the keys vary between submissions, the useful move is to flatten the
characteristics of every sample into one table and look at what the study
actually recorded.

In [7]:
samples = pd.DataFrame(
    {"sample": s.accession, "title": s.title, **s.channels[0].characteristics}
    for s in d.samples
)
samples.head()

,sample,title,cell type,donor age (years),resident tissue
0,GSM3148585,BC01_BLOOD1,CD45+ leukocytes,38,blood
1,GSM3148586,BC01_BLOOD3,CD45+ leukocytes,38,blood
2,GSM3148587,BC01_NORMAL1,CD45+ leukocytes,38,breast
3,GSM3148588,BC01_NORMAL2,CD45+ leukocytes,38,breast
4,GSM3148589,BC01_NORMAL3,CD45+ leukocytes,38,breast


From there the usual pandas work applies — count the levels of a variable,
then select the samples you want.

In [8]:
print(samples["resident tissue"].value_counts().to_dict())

tumour = samples[samples["resident tissue"] == "breast tumor"]
print(len(tumour), "tumour samples:", list(tumour["sample"])[:4])

{'breast tumor': 30, 'breast': 11, 'blood': 9, 'lymph node': 6}
30 tumour samples: ['GSM3148591', 'GSM3148592', 'GSM3148593', 'GSM3148594']


Samples carry their own supplementary files, separate from the project's.

In [9]:
for s in d.samples[:3]:
    for url in s.supplementary_data:
        print(s.accession, url.rsplit("/", 1)[-1])

GSM3148585 GSM3148585_BC01_BLOOD1_counts.csv.gz
GSM3148586 GSM3148586_BC01_BLOOD3_counts.csv.gz
GSM3148587 GSM3148587_BC01_NORMAL1_counts.csv.gz


`download_files` takes a bare list of URLs, which is how you fetch the
per-sample files for a subset you have selected:

```python
urls = [u for s in d.samples if s.accession in set(tumour["sample"])
        for u in s.supplementary_data]
sq.download_files(urls, Path("GSE114725/tumour"))
```

Any result set writes to CSV or converts to a DataFrame directly, without the
flattening above, when the raw record is what you want:

```python
d.samples.to_csv("samples.csv")
```

## experiments — the library preparations

An experiment describes how a library was made and on what instrument. A GEO
series holds none of its own; these come from the linked sequence archive.

In [10]:
e = d.experiments[0]
print(e.accession, "-", e.title)
print("strategy: ", e.library_strategy, "|", e.library_selection, "|", e.library_source)
print("layout:   ", e.library_layout)
print("platform: ", e.platform, "-", e.instrument_model)
print("samples:  ", e.samples)

SRX4108636 - GSM3148585: BC01_BLOOD1; Homo sapiens; RNA-Seq
strategy:  RNA-Seq | cDNA | TRANSCRIPTOMIC
layout:    PAIRED
platform:  ILLUMINA - Illumina HiSeq 2500
samples:   ['SRS3324288']


The summary that matters for most studies is which instruments were used, and
whether the libraries are consistent.

In [11]:
exps = d.experiments.to_df()
exps.groupby(["instrument_model", "library_strategy"]).size().rename("experiments")

instrument_model     library_strategy
Illumina HiSeq 2500  RNA-Seq             50
Illumina HiSeq 4000  RNA-Seq              6
Name: experiments, dtype: int64

## runs — the sequencing data

A run is one sequencing run of one library, and carries the URLs to its files.
`runs` reads every run, not the preview page the API returns by default.

In [12]:
r = d.runs[0]

print(r.run_accession, f"({r.library_layout})")
print("experiment:", r.experiment_accession)
print("study:     ", r.study_accession)
print()
for fmt, url in {
    "fastq": r.fastq_ftp,
    "sra": r.sra_ftp,
    "s3": r.ncbi_sra_lite_s3_url,
    "gcs": r.ncbi_sra_lite_gs_url,
}.items():
    if url:
        print(f"  {fmt:6} {url}")

SRR7191948 (PAIRED)
experiment: SRX4108636
study:      None

  fastq  ftp.sra.ebi.ac.uk/vol1/fastq/SRR719/008/SRR7191948/SRR7191948_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR719/008/SRR7191948/SRR7191948_2.fastq.gz


The FASTQ fields are semicolon-joined when a run has more than one file, which
is normal for paired-end data: the sizes and the MD5 sums line up with the
URLs.

In [13]:
print("urls: ", r.fastq_ftp.split(";"))
print("bytes:", r.fastq_bytes.split(";"))
print("md5:  ", r.fastq_md5.split(";"))

urls:  ['ftp.sra.ebi.ac.uk/vol1/fastq/SRR719/008/SRR7191948/SRR7191948_1.fastq.gz', 'ftp.sra.ebi.ac.uk/vol1/fastq/SRR719/008/SRR7191948/SRR7191948_2.fastq.gz']
bytes: ['4920822070', '5573314619']
md5:   ['833dc2cd36639a46f3b6c4b1931cc6e9', '1dc65935616cd401de2ab5ece75c4194']


Sum those sizes before you start a download. This study is larger than it
looks from the sample count.

In [14]:
total = sum(
    int(part)
    for run in d.runs
    for part in str(run.fastq_bytes or "").split(";")
    if part.strip().isdigit()
)
print(f"{len(d.runs)} runs, {total / 1e9:.1f} GB of FASTQ")
print("layouts:", pd.Series([run.library_layout for run in d.runs]).value_counts().to_dict())

173 runs, 772.3 GB of FASTQ
layouts: {'PAIRED': 173}


`download_study_runs_data` takes the runs you pass it, so a subset is just a
filtered list. Files are fetched in parallel and checked against the size and
MD5 above.

```python
first_two = StudyRunsResults(list(d.runs)[:2])
sq.download_study_runs_data(first_two, Path("reads"), mode="fastq")
```

The mode is `fastq`, `sra`, `sra_lite`, `s3`, or `gcs`. Not every run offers
every mode, so check before you commit to one.

In [15]:
pd.Series(
    {
        "fastq": sum(1 for run in d.runs if run.fastq_ftp),
        "sra": sum(1 for run in d.runs if run.sra_ftp),
        "s3": sum(1 for run in d.runs if run.ncbi_sra_lite_s3_url),
        "gcs": sum(1 for run in d.runs if run.ncbi_sra_lite_gs_url),
    },
    name="runs offering this format",
)

fastq    173
sra        0
s3         0
gcs        0
Name: runs offering this format, dtype: int64

## pubs — the publications

A dataset can be linked to more than one paper: the one that described it, and
later papers that reused the data. Each carries the journal metrics seqout
holds.

In [16]:
for p in d.pubs:
    print(p.title)
    print(f"  pmid {p.pmid} | doi {p.doi}")
    print(f"  {p.journal}, {p.pub_date} - {p.citation_count} citations")
    print(f"  authors: {(p.authors or '')[:80]}")

Single-Cell Map of Diverse Immune Phenotypes in the Breast Tumor Microenvironment.
  pmid 29961579 | doi 10.1016/j.cell.2018.05.060
  Cell, 2018 Aug 23 - 2129 citations
  authors: Elham Azizi, Ambrose J Carr, George Plitas, Andrew E Cornish, Catherine Konopack


The reverse lookup goes through `paper`, which takes a PubMed ID or a DOI and
returns every dataset linked to it.

In [17]:
pub = sq.paper(pmid=d.pubs[0].pmid)
print(pub.total_projects, "datasets linked to this paper")
pd.DataFrame(p.model_dump() for p in pub.projects[:5])

5 datasets linked to this paper


,accession,source,title,via
0,GSE114724,geo,Single-cell Map of Diverse Immune Phenotypes i...,study_publications
1,GSE114725,geo,Single-cell Map of Diverse Immune Phenotypes i...,study_publications
2,GSE114727,geo,Single-cell Map of Diverse Immune Phenotypes i...,study_publications
3,SRP148594,sra,Single-cell Map of Diverse Immune Phenotypes i...,study_publications
4,SRP148597,sra,Single-cell Map of Diverse Immune Phenotypes i...,study_publications


## links — the same data elsewhere

`links` lists the accessions other archives use for this data. A link with
`source` of `pmid` was matched through a shared publication rather than a
declared cross-reference, so it may not cover the same samples.

In [18]:
d.links.to_df()

,accession,link_type,source,via_pmid,title
0,GSE114727,SubSeries of,cross_ref_geo,None,None
1,PRJNA472383,BioProject,cross_ref_geo,None,None
2,SRP148597,SRA,cross_ref_geo,None,None
3,GSE114724,SuperSeries of,cross_ref_geo_2hop,None,None
4,PRJNA472387,BioProject,cross_ref_geo_2hop,None,None


## enriched — structured sample labels

seqout.org prepares normalized labels for the samples of some projects —
tissue, disease, cell type, assay, and their ontology terms. Coverage is
partial; the field is empty for a project that has not been processed, as
here.

In [19]:
print(len(d.enriched), "samples with enriched labels")

0 samples with enriched labels


For a project that has them, the result tabulates like any other field:

```python
d.enriched.to_df()[["sample", "tissue", "disease", "cell_type", "assay"]]
```

The command-line tool can also produce these locally with a small language
model — see `seqout --norm`, described in the
[normalization](../normalization.md) page.

## detail — the record for a child accession

When the accession you passed to `get` names a sample or a run rather than a
study, `detail` gives that record. For a study or a series it is `None`,
because `meta` already holds it.

In [20]:
run = sq.get("SRR7191509")
print(run.kind, "->", run.detail.run_accession, "in study", run.detail.study_accession)

sample = sq.get("GSM3148585")
print(sample.kind, "->", sample.detail.sample.title)
print("belongs to:", sample.detail.project.accession)

run -> SRR7191509 in study SRP148589


sample -> BC01_BLOOD1
belongs to: GSE114727


## Where the data sits

The last four fields report what `get` resolved. `kind` comes from the
accession pattern and costs no request; the others may need one.

In [21]:
for accession in ("GSE168652", "SRP310139", "GSM5155196", "SRR13927092"):
    x = sq.get(accession)
    print(f"{accession:12} kind={x.kind:10} project={x.project:12} geo={x.geo} sra={x.sra}")

GSE168652    kind=series     project=GSE168652    geo=GSE168652 sra=SRP310139


SRP310139    kind=study      project=SRP310139    geo=GSE168652 sra=SRP310139


GSM5155196   kind=sample     project=GSE168652    geo=GSE168652 sra=SRP310139


SRR13927092  kind=run        project=SRP310139    geo=GSE168652 sra=SRP310139


Those four accessions name the same dataset, so all of them arrive at the same
pair of archive records, and the fields above read the same whichever one you
start from.

GSE114725 is a reminder that this is not always so tidy. It is a GEO
SuperSeries, and its samples belong to the SubSeries beneath it, so
`sq.get("GSM3148585").project` is GSE114727 rather than GSE114725. The `links`
table above shows that relationship.

In [22]:
print(sq.get("GSM3148585").project, "- the SubSeries the sample belongs to")

GSE114727 - the SubSeries the sample belongs to


The [accessions and archives](03_accessions_and_archives.ipynb) notebook
covers how the resolution works, and what happens when it cannot.